[Guide Index](README.md)


# 06b Where Your Data Lives: On-Disk Stores and Remote Endpoints

Most examples in the StarLayer user guide use an in-memory `StarLayerGraph`. A `StarLayerGraph` wraps a `pyoxigraph.Store`, so the data is always held by Oxigraph in the same process (see [the Oxigraph-backed graph guide](02d-oxigraph-wrapper.ipynb)). This guide shows the two ways to go beyond memory:
- **An on-disk Oxigraph store**: `StarLayerGraph(path=...)` / `StarLayerDataset(path=...)` open a RocksDB store in a directory, so the data survives the process (section 1).
- **A remote SPARQL endpoint** (Oxigraph server, Fuseki, ...) can be read from a query with `SERVICE` (section 2). A graph object does not connect to a remote store, and StarLayer has no client for sending SPARQL updates to one.

There is no `backend=` argument and no `store=` for SQLAlchemy or other `rdflib` store plugins: a `StarLayerGraph` is always Oxigraph speaking RDF 1.2. To give an RDF 1.1-only system RDF 1.2 data, export it in RDF 1.1 form with `serialize(rdf_version="1.1")` (see [Serialization Formats](06a-serialization-formats.ipynb)).

## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01-getting-started.ipynb). Run cells from top to bottom; later cells reuse variables from earlier ones.

Section 1 needs only a writable directory. Section 2 queries a SPARQL endpoint that you run yourself; its cell skips cleanly if the endpoint is not reachable. The outputs saved in this notebook were produced without a server running, so section 2 shows the "skipping" message.

In [1]:
from rdflib import Namespace
from starlayer.graph import StarLayerGraph, TripleTerm

EX = Namespace("http://example.org/")

## 1. Using an on-disk store

`StarLayerGraph(path=...)` opens (or creates) an Oxigraph store in a directory, so the data survives the process. Only one process may write to a path at a time: drop the first object before opening the path again. The wrapper has no `open()` / `commit()` / `close()`; writes are applied as they are made (`g.store.flush()` forces them to disk, see [the Oxigraph-backed graph guide](02d-oxigraph-wrapper.ipynb)).

In [2]:
import tempfile

db_path = tempfile.mkdtemp()

writer = StarLayerGraph(path=db_path)
writer.bind("ex", EX)
writer.add_reification(EX.claim, TripleTerm(EX.bob, EX.knows, EX.carol))
writer.store.flush()
del writer  # release the single-writer lock on the directory

# fresh graph object, same directory - proves the data actually persisted
reader = StarLayerGraph(path=db_path)
reader.bind("ex", EX)
print(len(reader), "triple read back from disk")
print(reader.serialize(format="turtle", pretty=True))

1 triple read back from disk
@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:claim rdf:reifies <<( ex:bob ex:knows ex:carol )>> .



## 2. Reading from a remote SPARQL endpoint

A query run on a `StarLayerGraph` may contain a `SERVICE <endpoint> { ... }` block; Oxigraph sends that part to the endpoint over HTTP and joins the answer with the local data (SPARQL 1.1 Federated Query). It reads only. It works against Oxigraph server, Fuseki or any other SPARQL endpoint; the local graph here is empty. `SERVICE SILENT` returns no rows for the block instead of an error when the endpoint cannot be reached; without `SILENT` an unreachable endpoint raises an `OSError` (`ConnectionRefusedError`, `urllib.error.URLError`).

To write to a remote endpoint, send a SPARQL update with your own HTTP client; StarLayer has no function for it. Entailment on a remote endpoint (for example a Fuseki started with `--rdfs`) is a property of how the endpoint is configured: the endpoint answers with its own reasoning applied. For StarLayer's own reasoning on a `StarLayerGraph`, which needs no server, see the [Graph Inferencing guide](02b-graphs-inferencing.ipynb).

The example assumes an Oxigraph server at `http://localhost:7878/query` holding some `ex:Person` resources; your endpoint may differ.

In [3]:
ENDPOINT = "http://localhost:7878/query"

local = StarLayerGraph()
query = f"""
PREFIX ex: <http://example.org/>
SELECT ?person WHERE {{
  SERVICE <{ENDPOINT}> {{ ?person a ex:Person }}
}}
"""
try:
    print([str(row["person"]) for row in local.query(query)])
except OSError:
    print(f"skipping - no live instance at {ENDPOINT}")

skipping - no live instance at http://localhost:7878/query
